In [5]:
import pandas as pd
import numpy as np

SRC_PATH = "bordereaux_stc_status_avec_délais.xlsx"
OUT_PATH = "Test Depassement limites STC.xlsx"

xls = pd.read_excel(SRC_PATH, sheet_name=None)
COL_ALIASES = {
    "DAT_SORT": ["DAT_SORT", "DATE_SORT", "DATE_SORTIE", "DAT_SORTIE", "SORT_DATE", "DATE SORT"],
    "DATE_JOURNEE": ["DATE_JOURNEE", "DATE JO
                     URNEE", "JOURNEE_DATE", "DATE_JRN", "DATE JOURNÉE"],
    "COD_CENT": ["COD_CENT", "CODE_CENTRE", "CODE CENTRE", "CENTRE", "CODE_CENT"],
    "TOTAL": ["TOTAL", "MONTANT", "AMOUNT", "VALEUR", "TOTAL_TND", "TOTAL TTC", "TOTAL_TTC", "TOTAL_HT"],
}
REQUIRED = ["DAT_SORT", "DATE_JOURNEE", "COD_CENT", "TOTAL"]

def resolve_columns(df: pd.DataFrame) -> dict:
    df_cols = [str(c).strip() for c in df.columns]
    mapping = {}
    for target, candidates in COL_ALIASES.items():
        for cand in candidates:
            if cand in df_cols:
                mapping[target] = cand
                break
            low_df = [c.lower() for c in df_cols]
            if cand.lower() in low_df:
                mapping[target] = df_cols[low_df.index(cand.lower())]
                break
    return mapping

normalized = []
for sheet_name, sheet_df in xls.items():
    if sheet_df is None or sheet_df.empty:
        continue
    df = sheet_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    mapping = resolve_columns(df)

    out = pd.DataFrame()
    for col in REQUIRED:
        src = mapping.get(col)
        out[col] = df[src] if src in df.columns else np.nan
    out["__sheet__"] = sheet_name
    normalized.append(out)

data = pd.concat(normalized, ignore_index=True)

def to_num(x):
    if isinstance(x, str):
        x2 = x.replace(" ", "").replace("\xa0", "").replace(",", ".")
        try: return float(x2)
        except: return np.nan
    try: return float(x)
    except: return np.nan

data["DAT_SORT"] = pd.to_datetime(data["DAT_SORT"], errors="coerce")
data["DATE_JOURNEE"] = pd.to_datetime(data["DATE_JOURNEE"], errors="coerce")
data["TOTAL"] = data["TOTAL"].apply(to_num)
data["COD_CENT"] = data["COD_CENT"].astype(str).str.strip()

data["ECART_JOURS"] = (data["DAT_SORT"] - data["DATE_JOURNEE"]).dt.days
df_gt1 = data.loc[data["ECART_JOURS"] > 1, ["COD_CENT", "DAT_SORT", "DATE_JOURNEE", "TOTAL"]].copy()

agg = (
    df_gt1.groupby(["COD_CENT", "DAT_SORT"], dropna=False)
          .agg(CUMUL_ECART_GT1=("TOTAL", "sum"),
               NB_MVT=("TOTAL", "size"))
          .reset_index()
)

def type_centre(code):
    if code.startswith("R") or code.startswith("M"):
        return "Boutique"
    return "Franchise"

agg["TYPE_CENTRE"] = agg["COD_CENT"].apply(type_centre)

def is_anomaly(row):
    if row["TYPE_CENTRE"] == "Boutique":
        return row["CUMUL_ECART_GT1"] > 30000
    else:
        return row["CUMUL_ECART_GT1"] > 15000

agg["ANOMALIE"] = agg.apply(is_anomaly, axis=1)
anomalies_agg = agg.loc[agg["ANOMALIE"]].copy()

details = df_gt1.merge(anomalies_agg[["COD_CENT", "DAT_SORT"]], on=["COD_CENT", "DAT_SORT"], how="inner")
details = details.sort_values(["COD_CENT", "DAT_SORT", "DATE_JOURNEE"])

with pd.ExcelWriter(OUT_PATH, engine="xlsxwriter") as writer:
    agg.to_excel(writer, index=False, sheet_name="TCD_Complet")
    anomalies_agg.to_excel(writer, index=False, sheet_name="Anomalies")
    details.to_excel(writer, index=False, sheet_name="Details_Contributeurs")

print(f"Export OK : {OUT_PATH}")

Export OK : tcd_anomalies_par_DAT_SORT_type.xlsx


In [6]:
import pandas as pd
import numpy as np

SRC_PATH = "bordereaux_stc_status_avec_délais.xlsx"
OUT_PATH = "Test Depassement limites STC.xlsx"

xls = pd.read_excel(SRC_PATH, sheet_name=None)
COL_ALIASES = {
    "DAT_SORT": ["DAT_SORT", "DATE_SORT", "DATE_SORTIE", "DAT_SORTIE", "SORT_DATE", "DATE SORT"],
    "DATE_JOURNEE": ["DATE_JOURNEE", "DATE JOURNEE", "JOURNEE_DATE", "DATE_JRN", "DATE JOURNÉE"],
    "COD_CENT": ["COD_CENT", "CODE_CENTRE", "CODE CENTRE", "CENTRE", "CODE_CENT"],
    "TOTAL": ["TOTAL", "MONTANT", "AMOUNT", "VALEUR", "TOTAL_TND", "TOTAL TTC", "TOTAL_TTC", "TOTAL_HT"],
}
REQUIRED = ["DAT_SORT", "DATE_JOURNEE", "COD_CENT", "TOTAL"]

def resolve_columns(df: pd.DataFrame) -> dict:
    df_cols = [str(c).strip() for c in df.columns]
    mapping = {}
    for target, candidates in COL_ALIASES.items():
        for cand in candidates:
            if cand in df_cols:
                mapping[target] = cand
                break
            low_df = [c.lower() for c in df_cols]
            if cand.lower() in low_df:
                mapping[target] = df_cols[low_df.index(cand.lower())]
                break
    return mapping

normalized = []
for sheet_name, sheet_df in xls.items():
    if sheet_df is None or sheet_df.empty:
        continue
    df = sheet_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    mapping = resolve_columns(df)

    out = pd.DataFrame()
    for col in REQUIRED:
        src = mapping.get(col)
        out[col] = df[src] if src in df.columns else np.nan
    out["__sheet__"] = sheet_name
    normalized.append(out)

data = pd.concat(normalized, ignore_index=True)

def to_num(x):
    if isinstance(x, str):
        x2 = x.replace(" ", "").replace("\xa0", "").replace(",", ".")
        try:
            return float(x2)
        except:
            return np.nan
    try:
        return float(x)
    except:
        return np.nan

data["DAT_SORT"] = pd.to_datetime(data["DAT_SORT"], errors="coerce")
data["DATE_JOURNEE"] = pd.to_datetime(data["DATE_JOURNEE"], errors="coerce")
data["TOTAL"] = data["TOTAL"].apply(to_num)
data["COD_CENT"] = data["COD_CENT"].astype(str).str.strip()

cumuls = (
    data.groupby(["COD_CENT", "DAT_SORT"], dropna=False)["TOTAL"]
        .sum()
        .rename("CUMUL_PAR_JOUR")
        .reset_index()
)

data = data.merge(cumuls, on=["COD_CENT", "DAT_SORT"], how="left")

def type_centre(code: str) -> str:
    return "Boutique" if (isinstance(code, str) and (code.startswith("R") or code.startswith("M"))) else "Franchise"

data["TYPE_CENTRE"] = data["COD_CENT"].apply(type_centre)
data["SEUIL"] = np.where(data["TYPE_CENTRE"] == "Boutique", 30000.0, 15000.0)
data["DEPASSE_SEUIL"] = data["CUMUL_PAR_JOUR"] > data["SEUIL"]

delta_days = (data["DATE_JOURNEE"] - data["DAT_SORT"]).dt.days

cond_depasse = (data["DATE_JOURNEE"] == data["DAT_SORT"])
cond_pas_depasse = (delta_days >= 1)

data["CONDITION_OK"] = np.where(data["DEPASSE_SEUIL"], cond_depasse, cond_pas_depasse)

data["DELTA_JOURS"] = delta_days

violations = (
    data.loc[~data["CONDITION_OK"], [
        "COD_CENT", "TYPE_CENTRE", "DAT_SORT", "DATE_JOURNEE",
        "TOTAL", "CUMUL_PAR_JOUR", "SEUIL", "DEPASSE_SEUIL", "DELTA_JOURS", "__sheet__"
    ]]
    .sort_values(["COD_CENT", "DAT_SORT", "DATE_JOURNEE"], kind="mergesort")
)

agg = (
    data.groupby(["COD_CENT", "TYPE_CENTRE", "DAT_SORT", "DEPASSE_SEUIL"], dropna=False)
        .agg(
            CUMUL_PAR_JOUR=("CUMUL_PAR_JOUR", "first"),
            SEUIL=("SEUIL", "first"),
            NB_LIGNES=("TOTAL", "size"),
            MONTANT_TOTAL=("TOTAL", "sum"),
            NB_VIOLATIONS=("CONDITION_OK", lambda s: (~s).sum()),
        )
        .reset_index()
        .sort_values(["COD_CENT", "DAT_SORT"])
)

with pd.ExcelWriter(OUT_PATH, engine="xlsxwriter") as writer:
    data.to_excel(writer, index=False, sheet_name="Donnees_Avec_Regles")
    agg.to_excel(writer, index=False, sheet_name="Synthese_Jour")
    violations.to_excel(writer, index=False, sheet_name="Violations")

print(f"Export OK : {OUT_PATH}")

Export OK : Test Depassement limites STC.xlsx


In [8]:
import pandas as pd
import numpy as np

SRC_PATH = "bordereaux_stc_status_avec_délais.xlsx"
OUT_PATH = "Testttt Depassement limites STC.xlsx"

xls = pd.read_excel(SRC_PATH, sheet_name=None)
COL_ALIASES = {
    "DAT_SORT": ["DAT_SORT", "DATE_SORT", "DATE_SORTIE", "DAT_SORTIE", "SORT_DATE", "DATE SORT"],
    "DATE_JOURNEE": ["DATE_JOURNEE", "DATE JOURNEE", "JOURNEE_DATE", "DATE_JRN", "DATE JOURNÉE"],
    "COD_CENT": ["COD_CENT", "CODE_CENTRE", "CODE CENTRE", "CENTRE", "CODE_CENT"],
    "TOTAL": ["TOTAL", "MONTANT", "AMOUNT", "VALEUR", "TOTAL_TND", "TOTAL TTC", "TOTAL_TTC", "TOTAL_HT"],
}
REQUIRED = ["DAT_SORT", "DATE_JOURNEE", "COD_CENT", "TOTAL"]

def resolve_columns(df: pd.DataFrame) -> dict:
    df_cols = [str(c).strip() for c in df.columns]
    mapping = {}
    for target, candidates in COL_ALIASES.items():
        for cand in candidates:
            if cand in df_cols:
                mapping[target] = cand
                break
            low_df = [c.lower() for c in df_cols]
            if cand.lower() in low_df:
                mapping[target] = df_cols[low_df.index(cand.lower())]
                break
    return mapping

normalized = []
for sheet_name, sheet_df in xls.items():
    if sheet_df is None or sheet_df.empty:
        continue
    df = sheet_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    mapping = resolve_columns(df)

    out = pd.DataFrame()
    for col in REQUIRED:
        src = mapping.get(col)
        out[col] = df[src] if src in df.columns else np.nan
    out["__sheet__"] = sheet_name
    normalized.append(out)

if not normalized:
    raise ValueError("Aucune feuille exploitable dans le fichier source.")

data = pd.concat(normalized, ignore_index=True)

def to_num(x):
    if isinstance(x, str):
        x2 = x.replace(" ", "").replace("\xa0", "").replace(",", ".")
        try:
            return float(x2)
        except Exception:
            return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan

data["DAT_SORT"] = pd.to_datetime(data["DAT_SORT"], errors="coerce")
data["DATE_JOURNEE"] = pd.to_datetime(data["DATE_JOURNEE"], errors="coerce")
data["TOTAL"] = data["TOTAL"].apply(to_num)
data["COD_CENT"] = data["COD_CENT"].astype(str).str.strip()

data["DAT_SORT"] = data["DAT_SORT"].dt.normalize()
data["DATE_JOURNEE"] = data["DATE_JOURNEE"].dt.normalize()

cumuls = (
    data.groupby(["COD_CENT", "DAT_SORT"], dropna=False)["TOTAL"]
        .sum()
        .rename("CUMUL_PAR_JOUR")
        .reset_index()
)
data = data.merge(cumuls, on=["COD_CENT", "DAT_SORT"], how="left")

def type_centre(code: str) -> str:
    return "Boutique" if (isinstance(code, str) and (code.startswith("R") or code.startswith("M"))) else "Franchise"

data["TYPE_CENTRE"] = data["COD_CENT"].apply(type_centre)
data["SEUIL"] = np.where(data["TYPE_CENTRE"] == "Boutique", 30000.0, 15000.0)
data["DEPASSE_SEUIL"] = data["CUMUL_PAR_JOUR"] > data["SEUIL"]

delta_days = (data["DATE_JOURNEE"] - data["DAT_SORT"]).dt.days
cond_depasse = (data["DATE_JOURNEE"] == data["DAT_SORT"])
cond_pas_depasse = (delta_days >= 1)
data["CONDITION_OK"] = np.where(data["DEPASSE_SEUIL"], cond_depasse, cond_pas_depasse)

data["DELTA_JOURS"] = delta_days

first_journee = (
    data.groupby(["COD_CENT", "DAT_SORT"], dropna=False)["DATE_JOURNEE"]
        .min()
        .rename("PREMIERE_DATE_JOURNEE")
        .reset_index()
)
first_journee["NB_JOURS_ANOMALIE"] = (
    (first_journee["PREMIERE_DATE_JOURNEE"] - first_journee["DAT_SORT"]).dt.days
)

violations = (
    data.loc[~data["CONDITION_OK"], [
        "COD_CENT", "TYPE_CENTRE", "DAT_SORT", "DATE_JOURNEE",
        "TOTAL", "CUMUL_PAR_JOUR", "SEUIL", "DEPASSE_SEUIL", "DELTA_JOURS", "__sheet__"
    ]]
    .sort_values(["COD_CENT", "DAT_SORT", "DATE_JOURNEE"], kind="mergesort")
)

violations = violations.merge(
    first_journee[["COD_CENT", "DAT_SORT", "PREMIERE_DATE_JOURNEE", "NB_JOURS_ANOMALIE"]],
    on=["COD_CENT", "DAT_SORT"],
    how="left",
)

agg = (
    data.groupby(["COD_CENT", "TYPE_CENTRE", "DAT_SORT", "DEPASSE_SEUIL"], dropna=False)
        .agg(
            CUMUL_PAR_JOUR=("CUMUL_PAR_JOUR", "first"),  # identique sur le groupe
            SEUIL=("SEUIL", "first"),
            NB_LIGNES=("TOTAL", "size"),
            MONTANT_TOTAL=("TOTAL", "sum"),
            NB_VIOLATIONS=("CONDITION_OK", lambda s: (~s).sum()),
        )
        .reset_index()
        .sort_values(["COD_CENT", "DAT_SORT"])
)

agg = agg.merge(
    first_journee[["COD_CENT", "DAT_SORT", "NB_JOURS_ANOMALIE", "PREMIERE_DATE_JOURNEE"]],
    on=["COD_CENT", "DAT_SORT"],
    how="left",
)

with pd.ExcelWriter(OUT_PATH, engine="xlsxwriter") as writer:
    data.to_excel(writer, index=False, sheet_name="Donnees_Avec_Regles")
    agg.to_excel(writer, index=False, sheet_name="Synthese_Jour")
    violations.to_excel(writer, index=False, sheet_name="Violations")

print(f"Export OK : {OUT_PATH}")

Export OK : Testttt Depassement limites STC.xlsx
